In [1]:
cd ../codes

/home/ec2-user/StableDiffusionReconstruction/codes


/home/ec2-user/StableDiffusionReconstruction/.venv/lib/python3.11/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [ ]:
import h5py
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
from PIL import Image
from utils.nsd_access.nsda import NSDAccess

# ======================================
# USER CONFIG
# ======================================

subject = "subj01"
start_idx = 0       # inclusive
end_idx   = 981      # exclusive
images_per_row = 10

# ======================================
# 1) Load NSD experiment design
# ======================================

nsd_expdesign = scipy.io.loadmat('../nsd/nsddata/experiments/nsd/nsd_expdesign.mat')

# IMPORTANT: subtract 1 exactly as script does
sharedix = nsd_expdesign['sharedix'] - 1  

stims_ave = np.load(f'../mrifeat/{subject}/{subject}_stims_ave.npy')

# Reproduce train/test split EXACTLY
tr_idx = np.zeros_like(stims_ave)
for idx, s in enumerate(stims_ave):
    if s in sharedix:
        tr_idx[idx] = 0
    else:
        tr_idx[idx] = 1

test_positions = np.where(tr_idx == 0)[0]

# ======================================
# 2) Load NSD HDF5 stimuli file
# ======================================

nsda = NSDAccess('../nsd/')
sf = h5py.File(nsda.stimuli_file, 'r')
sdataset = sf.get('imgBrick')   # EXACT same dataset used in script

# ======================================
# 3) Collect images EXACTLY as script
# ======================================

images = []
titles = []

for imgidx in range(start_idx, end_idx):

    # EXACT line from your script
    imgidx_te = test_positions[imgidx]
    idx73k = stims_ave[imgidx_te]

    # EXACT image extraction
    img_array = np.squeeze(
        sdataset[idx73k, :, :, :]
    ).astype(np.uint8)

    img = Image.fromarray(img_array)

    images.append(img)
    titles.append(f"imgidx={imgidx}\nidx73k={idx73k}")

# ======================================
# 4) Plot Mosaic
# ======================================

num_images = len(images)
rows = int(np.ceil(num_images / images_per_row))

plt.figure(figsize=(20, 3 * rows))

for i, (img, title) in enumerate(zip(images, titles)):
    plt.subplot(rows, images_per_row, i + 1)
    plt.imshow(img)
    plt.title(title, fontsize=9)
    plt.axis("off")

plt.tight_layout()
plt.show()

# Optional: close file after use
sf.close()